# Lesson 11 — The End-to-End Machine Learning Pipeline

**What this notebook does:** it takes one job — sorting support tickets into a category — and walks it through all five stages of machine learning, one at a time: **data**, **features**, **train**, **evaluate**, **use**.

The model we build is a very simple one you already understand (counting words), and that is on purpose. The point of this lesson is the *five-stage journey*, not a clever algorithm. Fancier algorithms come later, but they all drop into these same five stages.

Every code cell below has short comments on each line, so you can read the code straight through. The markdown before each cell explains the bigger *why*.

## The five stages

Almost every machine-learning project follows the same five steps in order:

1. **Data** — gather examples that already have the right answer attached (a ticket plus the category it belongs to).
2. **Features** — turn each example into numbers or simple pieces the computer can work with.
3. **Train** — but first set some examples aside; then let the model learn its settings from the rest.
4. **Evaluate** — check how well it does on the examples we set aside and never let it study.
5. **Use** — point the finished model at a brand-new example that has no answer yet.

We do each stage in its own cell below.

## Stage 1 — Data

Machine learning starts with **examples that already have the right answer attached**. Ours are support tickets, and each one is paired with the category a human decided it belongs to: **shipping**, **billing**, or **account**. We have 12 tickets, four of each category.

Because every ticket comes with its correct answer, the computer can learn by comparing its own guesses against those answers. Learning from answered examples like this is called **supervised learning** (we met it in Lesson 02 — think of the answers as a teacher grading each guess).

The order of the rows below is deliberate: three of each category first, then one of each at the bottom. That way the first 9 rows can be the learning set and the last 3 the testing set.

In [1]:
import pandas as pd  # pandas is our tool for working with tables (Lesson 06)

# Build a table of tickets. Each row is a pair: (ticket text, its category).
tickets = pd.DataFrame([
    ("Where is my package it has not arrived yet", "shipping"),
    ("My delivery is late and tracking has not updated", "shipping"),
    ("When will my order ship out", "shipping"),
    ("I was charged twice for my order please refund", "billing"),
    ("There is a wrong charge on my card", "billing"),
    ("I need a refund for a payment I did not make", "billing"),
    ("I cannot log in to my account", "account"),
    ("How do I reset my password", "account"),
    ("My login is not working and I am locked out", "account"),
    ("My package still has not arrived", "shipping"),
    ("Why was I charged twice", "billing"),
    ("I forgot my password and cannot log in", "account"),
], columns=["text", "category"])  # name the two columns

tickets  # show the finished table


,text,category
0,Where is my package it has not arrived yet,shipping
1,My delivery is late and tracking has not updated,shipping
2,When will my order ship out,shipping
3,I was charged twice for my order please refund,billing
4,There is a wrong charge on my card,billing
5,I need a refund for a payment I did not make,billing
6,I cannot log in to my account,account
7,How do I reset my password,account
8,My login is not working and I am locked out,account
9,My package still has not arrived,shipping


## Stage 2 — Features

A computer cannot read a sentence the way we do. It works with numbers and small, separate pieces. So step two is to chop each ticket into the plain words it is made of. "My package is late" becomes a list: `['my', 'package', 'is', 'late']`.

Chopping text into a list of words like this has a name: **tokenising**, and each single word is called a **token**. For now, a token is just one lowercase word. We make everything lowercase first so that "My" and "my" count as the *same* word, not two different ones.

These word-pieces are our **features** — the simple, countable things the model will actually look at.

In [2]:
# A small helper: turn one piece of text into a list of lowercase words
def tokenize(text):
    return text.lower().split()  # .lower() lowercases it, .split() cuts it at spaces

# Do the same lowercasing + splitting to every row of the text column at once,
# and save the word-lists in a new column called "tokens".
tickets["tokens"] = tickets["text"].str.lower().str.split()  # .str applies text ops to the whole column

tickets[["text", "category", "tokens"]]  # show the text, its category, and its word-list


,text,category,tokens
0,Where is my package it has not arrived yet,shipping,"[where, is, my, package, it, has, not, arrived..."
1,My delivery is late and tracking has not updated,shipping,"[my, delivery, is, late, and, tracking, has, n..."
2,When will my order ship out,shipping,"[when, will, my, order, ship, out]"
3,I was charged twice for my order please refund,billing,"[i, was, charged, twice, for, my, order, pleas..."
4,There is a wrong charge on my card,billing,"[there, is, a, wrong, charge, on, my, card]"
5,I need a refund for a payment I did not make,billing,"[i, need, a, refund, for, a, payment, i, did, ..."
6,I cannot log in to my account,account,"[i, cannot, log, in, to, my, account]"
7,How do I reset my password,account,"[how, do, i, reset, my, password]"
8,My login is not working and I am locked out,account,"[my, login, is, not, working, and, i, am, lock..."
9,My package still has not arrived,shipping,"[my, package, still, has, not, arrived]"


## Stage 3 — Train (but first, hold out a test set)

The most important rule in all of machine learning: **judge the model on tickets it has never seen.** If we let it study all 12 tickets and then test it on those same 12, a model that just memorised them would look perfect — yet we would have learned nothing about how it handles a *new* ticket tomorrow.

So before any learning happens, we cut the pile in two: a **training set** the model is allowed to learn from (our first 9), and a **test set** we lock away and only look at at the very end (our last 3). Lesson 12 is all about this split, so here we keep it short.

In [3]:
train = tickets.iloc[:9]  # iloc picks rows by position; [:9] = the first 9 rows -> learn from these
test = tickets.iloc[9:]   # [9:] = row 9 to the end = the last 3 rows -> hidden until testing

print("Training tickets:", len(train))  # expect 9
print("Test tickets:", len(test))       # expect 3


Training tickets: 9
Test tickets: 3


### Now train: learn one word-count table per category

"Training" can sound like a black box, so here is exactly what happens, with no magic. For each category, we go through its training tickets and **count how many times every word appears**. That gives three little tally sheets — one for shipping, one for billing, one for account. Those three tally sheets *are* what the model "learned."

The numbers a model learns are called its **parameters**. Notice we never typed these numbers ourselves — the data filled them in. That is the whole difference from the hand-typed weights we used back in Lesson 01. Words like "refund" pile up under billing, "password" under account, "tracking" under shipping — so each word quietly becomes a clue that points at one category.

In [7]:
from collections import Counter  # Counter tallies how many times each word appears

# Learn a word-count tally sheet for each category from the training tickets
def train_word_counts(train_df):
    counts = {}                            # will map: category name -> its Counter tally
    for _, row in train_df.iterrows():     # go through the training tickets one by one
        counts.setdefault(row["category"], Counter())  # new category? give it a blank tally
        counts[row["category"]].update(row["tokens"])
    
    # print(counts)  # add this ticket's words to that tally
    return counts                          # three finished tallies = the learned model

model = train_word_counts(train)           # run training on the training set only

# Peek at the 4 most common words each category learned
for category, counter in model.items():
    print(category, "->", counter.most_common(4))


shipping -> [('my', 3), ('is', 2), ('has', 2), ('not', 2)]
billing -> [('i', 3), ('a', 3), ('for', 2), ('my', 2)]
account -> [('i', 3), ('my', 3), ('cannot', 1), ('log', 1)]


### How the model makes a guess

To label a ticket, we ask each category one question: "how well do this ticket's words match the words you learned?" We answer it by adding up, for every word in the ticket, how many times that word appeared in that category during training. Whichever category gets the biggest total wins.

Everyday words like "my" or "the" appear in every category, so they add about the same amount to all three totals and roughly cancel out. The words that belong to just one category — "refund", "password", "tracking" — are what actually decide the winner.

In [9]:
# Guess a ticket's category from its words and the learned tallies
def predict(tokens, model):
    scores = {}                              # one running score per category
    for category, counter in model.items():  # check the ticket against each category
        # add up how often the ticket's words appear in this category (0 if a word was never seen)
        scores[category] = sum(counter.get(word, 0) for word in tokens)
    return max(scores, key=scores.get), scores  # winner = highest score; also return all scores

predict("my package still has not arrived".split(), model)  # try it on the first test ticket


('shipping', {'shipping': 9, 'billing': 3, 'account': 4})

## Stage 4 — Evaluate

Now we unlock the test set — the 3 tickets the model never studied — and count how many it gets right. The fraction it gets right is called **accuracy**. Getting 3 out of 3 is an accuracy of `3 / 3 = 1.0`, meaning 100%.

But a single number can trick you. So we also work out a **baseline**: the score you would get from the laziest possible strategy — "always guess whichever category was most common in the training set." If a real model cannot beat that lazy guess, it is not worth keeping. The baseline is the bar to clear.

In [10]:
# Count what fraction of a table's tickets the model labels correctly
def accuracy(df, model):
    correct = 0                             # how many we've gotten right so far
    for _, row in df.iterrows():            # go through each ticket
        prediction, _ = predict(row["tokens"], model)  # the model's guess (ignore the scores)
        if prediction == row["category"]:   # did the guess match the true category?
            correct += 1                    # yes -> count it
    return correct / len(df)                # fraction right = accuracy

test_accuracy = accuracy(test, model)       # score on the held-out test set

# Baseline: what if we lazily guessed the most common training category every time?
majority_class = train["category"].value_counts().idxmax()      # name of the most common category
baseline_accuracy = (test["category"] == majority_class).mean()  # fraction that lazy guess gets right

print(f"Model accuracy on held-out test:      {test_accuracy:.2f}")
print(f"Baseline (always guess '{majority_class}'): {baseline_accuracy:.2f}")


Model accuracy on held-out test:      1.00
Baseline (always guess 'shipping'): 0.33


## Stage 5 — Use

This is the payoff: take the finished, tested model and point it at a brand-new ticket that has no label yet. This is exactly how it would run inside the assistant — a ticket comes in, the model tags it with a category.

In [11]:
new_ticket = "My package has not shipped and tracking shows it is late"  # a brand-new ticket

prediction, scores = predict(tokenize(new_ticket), model)  # split into words, then score each category

print("Ticket:", new_ticket)
print("Predicted category:", prediction)  # the winning category
print("Scores:", scores)                   # the score behind each category


Ticket: My package has not shipped and tracking shows it is late
Predicted category: shipping
Scores: {'shipping': 14, 'billing': 4, 'account': 6}


## The loop, not the line

Notice the shape: **data → features → train → evaluate → use.** If evaluation had been poor, we would not ship the model — we would loop back: collect more tickets, build better features, or try a stronger model, then evaluate again. That loop is most of the real job. The pieces get fancier across this course (real algorithms, careful splits, better scoring), but this skeleton never changes.

One honest weakness is visible right here: our model only credits words it saw during training, so a ticket that says "parcel" instead of "package" earns no shipping points for that word. Fixing that kind of blind spot is what the rest of Phase 2 is about.